In [ ]:

import gc
import json
import sys
from collections import Counter
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing.pipeline import (
    stratified_split,
    fit_scaler,
    apply_scaler,
    smote_train_only,
    remove_invalid_negative_rows,
)

LABEL = "Attack Type"
NORMAL = "Normal Traffic"
SEED = 42

df = pd.read_parquet(
    ROOT / "data" / "interim" / "cleaned.parquet"
)

df, removed = remove_invalid_negative_rows(df)

print("Invalid negative rows removed:", removed)
print("Remaining records:", len(df))

assert removed == 2909
assert len(df) == 2517681
assert df["Attack Type"].nunique() == 7

schema = json.loads(
    (ROOT / "data" / "schema" / "feature_schema.json").read_text(
        encoding="utf-8"
    )
)

FEATURE_COLUMNS = [
    feature["name"]
    for feature in schema["features"]
]

DROP_FEATURES = {
    "Active Mean",
    "Active Max",
    "Active Min",
    "Idle Mean",
    "Idle Max",
    "Idle Min",
}

assert len(FEATURE_COLUMNS) == 46
assert "Destination Port" in FEATURE_COLUMNS
assert DROP_FEATURES.isdisjoint(FEATURE_COLUMNS)

X = df[FEATURE_COLUMNS].copy()
y = df[LABEL]

print("Selected features:", X.shape[1])
print("Dataset shape:", X.shape)
print("Destination Port retained:", "Destination Port" in X.columns)
print(X.shape, y.shape)

del df
gc.collect()


In [ ]:

le = LabelEncoder()

y_enc = pd.Series(
    le.fit_transform(y),
    index=y.index,
    name="label"
)

mapping = {
    int(i): str(name)
    for i, name in enumerate(le.classes_)
}

print(mapping)

(ROOT / "models").mkdir(exist_ok=True)

joblib.dump(
    le,
    ROOT / "models" / "label_encoder.pkl"
)


import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing.pipeline import (
    stratified_split,
    fit_scaler,
    apply_scaler,
    smote_train_only,
)



In [ ]:

X_train, X_val, X_test, y_train, y_val, y_test = stratified_split(
    X, y_enc, seed=SEED
)

for name, part in [
    ("train", y_train),
    ("val", y_val),
    ("test", y_test),
]:
    share = part.value_counts(normalize=True).sort_index().round(4)
    print(name, len(part), dict(share))

assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)

print("Split checks passed")


for name, part in [
    ("train", y_train),
    ("val", y_val),
    ("test", y_test)
]:
    share = part.value_counts(normalize=True).sort_index().round(4)
    print(name, len(part), dict(share))

assert not (set(X_train.index) & set(X_val.index))
assert not (set(X_train.index) & set(X_test.index))
assert not (set(X_val.index) & set(X_test.index))

print("Split checks passed")


In [ ]:

INTERIM = ROOT / "data" / "interim"

for name, Xp, yp in [
    ("train", X_train, y_train),
    ("val", X_val, y_val),
    ("test", X_test, y_test)
]:
    out = Xp.copy()
    out["label"] = yp
    out.to_parquet(
        INTERIM / f"split_{name}_unscaled.parquet",
        index=False
    )
    print(f"Saved split_{name}_unscaled.parquet: {out.shape}")

del X, y, y_enc
gc.collect()


In [ ]:

schema = json.loads(
    (ROOT / "data" / "schema" / "feature_schema.json").read_text(
        encoding="utf-8"
    )
)

schema_cols = [f["name"] for f in schema["features"]]

assert list(X_train.columns) == schema_cols, (
    "Columns differ from feature_schema.json"
)

print("Columns match feature_schema.json")
print("Feature count:", len(schema_cols))


In [ ]:

scaler = fit_scaler(X_train)

X_train_s = apply_scaler(scaler, X_train)
X_val_s = apply_scaler(scaler, X_val)
X_test_s = apply_scaler(scaler, X_test)

joblib.dump(scaler, ROOT / "models" / "scaler.pkl")

scaler_metadata = {
    "feature_names": scaler.feature_names_in_.tolist(),
    "mean": scaler.mean_.tolist(),
    "scale": scaler.scale_.tolist(),
    "variance": scaler.var_.tolist(),
    "n_features": int(scaler.n_features_in_),
    "n_samples_seen": int(scaler.n_samples_seen_),
}

with open(ROOT / "models" / "scaler.json", "w", encoding="utf-8") as f:
    json.dump(scaler_metadata, f, indent=2)

print(
    "Largest train column average (should be about 0):",
    X_train_s.mean().abs().max()
)

print("Scaler and scaler.json saved successfully")

del X_train, X_val, X_test
gc.collect()


In [ ]:

train_counts = Counter(y_train)

print("Train class counts:", dict(sorted(train_counts.items())))

FLOOR = 10000

strategy = {
    cls: FLOOR
    for cls, count in train_counts.items()
    if count < FLOOR
}

print("Classes raised to", FLOOR, ":", strategy)

X_res, y_res = smote_train_only(
    X_train_s,
    y_train,
    floor=FLOOR,
    seed=SEED
)

print("After SMOTE:", dict(sorted(Counter(y_res).items())))

print(
    "Train rows",
    len(X_train_s),
    "->",
    len(X_res),
    "| val and test untouched:",
    len(X_val_s),
    len(X_test_s)
)


In [ ]:

PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

def save(Xp, yp, name):
    out = Xp.reset_index(drop=True).copy()
    out["label"] = np.asarray(yp)
    out.to_parquet(PROC / name, index=False)
    print(name, out.shape)

save(X_train_s, y_train, "train_scaled.parquet")
save(X_res, y_res, f"train_smote_floor{FLOOR}.parquet")
save(X_val_s, y_val, "val_scaled.parquet")
save(X_test_s, y_test, "test_scaled.parquet")

del X_train_s, X_val_s, X_test_s, X_res
gc.collect()
